<a href="https://colab.research.google.com/github/keenanpepper/beatdrummer/blob/gh-pages/CleanGPTNeo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.9 MB

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets
from datasets import load_dataset
import tiktoken
import os
import numpy as np
from tqdm import tqdm
import math

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class GPTNeoConfig:
    def __init__(self, vocab_size=50257, hidden_size=64, num_layers=8, num_heads=16,
                 max_position_embeddings=2048, window_size=256):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size  # refers to residual stream width. not number of neurons in MLP, that is 4x more
        self.num_layers = num_layers    # number of complete transformer blocks
        self.num_heads = num_heads
        self.max_position_embeddings = max_position_embeddings
        self.window_size = window_size  # only used in local attention layers - if all attention is global then this is unused
        self.attention_layers = ["global", "local"] * (num_layers // 2)

In [4]:
class Attention(nn.Module):
    def __init__(self, config, layer_id):
        super().__init__()
        self.is_local = (config.attention_layers[layer_id] == "local")
        self.num_heads = config.num_heads
        self.hidden_size = config.hidden_size
        self.head_dim = config.hidden_size // config.num_heads

        self.attention = nn.ModuleDict(dict(
            k_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False),
            v_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False),
            q_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False),
            out_proj = nn.Linear(config.hidden_size, config.hidden_size)
        ))

    def forward(self, x):
        batch_size, seq_len, _ = x.size()

        q = self.attention.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.attention.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.attention.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Intentionally don't divide by sqrt(self.num_heads) here, since the reference implementation
        # for which the pretrained weights were trained doesn't have that
        scores = torch.matmul(q, k.transpose(-1, -2))

        if self.is_local:
            # Local attention
            local_mask = torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device)
            local_mask = torch.triu(local_mask, diagonal=1) | torch.tril(local_mask, diagonal=-config.window_size)
            scores = scores.masked_fill(local_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        else:
            # Global attention
            causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device), diagonal=1)
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, v)

        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        return self.attention.out_proj(context)

In [27]:
class NewGELUActivation(nn.Module):
    """
    Implementation of the GELU activation function currently in Google BERT repo (identical to OpenAI GPT). Also see
    the Gaussian Error Linear Units paper: https://arxiv.org/abs/1606.08415
    """

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return 0.5 * input * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (input + 0.044715 * torch.pow(input, 3.0))))

In [28]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.hidden_size, 4 * config.hidden_size)
        self.c_proj = nn.Linear(4 * config.hidden_size, config.hidden_size)
#        self.act = nn.GELU()
        self.act = NewGELUActivation()

    def forward(self, x):
        return self.c_proj(self.act(self.c_fc(x)))

In [29]:
class GPTNeoBlock(nn.Module):
    def __init__(self, config, layer_id):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.hidden_size, eps=1e-5)
        self.attn = Attention(config, layer_id)
        self.ln_2 = nn.LayerNorm(config.hidden_size, eps=1e-5)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [30]:
class GPTNeo(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.hidden_size),
            wpe = nn.Embedding(config.max_position_embeddings, config.hidden_size),
            h = nn.ModuleList([GPTNeoBlock(config, i) for i in range(config.num_layers)]),
            ln_f = nn.LayerNorm(config.hidden_size, eps=1e-5)
        ))
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # tie weights
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, input_ids, targets=None):
        device = input_ids.device
        b, t = input_ids.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0)

        tok_emb = self.transformer.wte(input_ids)
        pos_emb = self.transformer.wpe(pos)

        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            # Reshape logits to (batch_size * sequence_length, vocab_size)
            logits_view = logits.view(-1, logits.size(-1))
            # Reshape targets to (batch_size * sequence_length,)
            targets_view = targets.view(-1)
            # Compute cross entropy loss
            loss = F.cross_entropy(logits_view, targets_view)

        return logits, loss

In [31]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

ts1m = AutoModelForCausalLM.from_pretrained('roneneldan/TinyStories-1M')

ts3m = AutoModelForCausalLM.from_pretrained('roneneldan/TinyStories-3M')

In [32]:
config = GPTNeoConfig(hidden_size=128)
model = GPTNeo(config)
model.load_state_dict(ts3m.state_dict())

<All keys matched successfully>

In [33]:
# Forward pass without targets
input_ids = torch.randint(0, config.vocab_size, (1, 100))
logits, loss = model(input_ids)
print("Logits shape:", logits.shape)  # Should be (1, 100, 50257)
print("Loss:", loss)  # Should be None

# Forward pass with targets
targets = torch.randint(0, config.vocab_size, (1, 100))
logits, loss = model(input_ids, targets)
print("Logits shape:", logits.shape)  # Should be (1, 100, 50257)
print("Loss:", loss)  # Should be a tensor with a single value

Logits shape: torch.Size([1, 100, 50257])
Loss: None
Logits shape: torch.Size([1, 100, 50257])
Loss: tensor(15.6283, grad_fn=<NllLossBackward0>)


In [11]:
ds = datasets.load_dataset("tdooms/TinyStories")

In [12]:
def prepare_data(dataset_name="tdooms/TinyStories", split="train", separator_token="<|endoftext|>", output_file="train.bin"):
    # Initialize tokenizer
    enc = tiktoken.get_encoding("gpt2")
    separator_token_id = enc.encode_single_token(separator_token)

    def process(example):
        ids = enc.encode_ordinary(example['text'])  # encode_ordinary ignores any special tokens
        ids.append(separator_token_id)  # Add separator token at the end of each example
        return {'ids': ids, 'len': len(ids)}

    # Load and process the dataset
    if not os.path.exists(output_file):
        print(f"Processing {dataset_name} dataset...")
        ds = load_dataset(dataset_name, split=split)

        tokenized = ds.map(
            process,
            remove_columns=['text'],
            desc="Tokenizing the dataset",
            num_proc=8,
        )

        # Concatenate all ids into one large file
        arr_len = np.sum(tokenized['len'], dtype=np.uint64)
        dtype = np.uint16  # Can use uint16 since enc.max_token_value == 50256 is < 2**16
        arr = np.memmap(output_file, dtype=dtype, mode='w+', shape=(arr_len,))

        total_batches = 1024
        idx = 0

        for batch_idx in tqdm(range(total_batches), desc=f'Writing {output_file}'):
            # Batch together samples for faster write
            batch = tokenized.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        arr.flush()
        print(f"Dataset processed and saved to {output_file}")
    else:
        print(f"{output_file} already exists. Skipping processing.")

    # Load the processed data
    data = np.memmap(output_file, dtype=np.uint16, mode='r')
    return data, enc

In [13]:
data, tokenizer = prepare_data(separator_token="<|endoftext|>")
print(f"Processed data shape: {data.shape}")
print(f"First 10 tokens: {data[:10]}")
print(f"Last 10 tokens: {data[-10:]}")

# Decode a small portion to verify
sample = data[:10000]
decoded = tokenizer.decode(sample.tolist())
print("\nSample decoded text:")
print(decoded[:5000] + "...")  # Print first 500 characters

train.bin already exists. Skipping processing.
Processed data shape: (396967656,)
First 10 tokens: [ 8888    11 19919   373   845  6568    13   679   373  1016]
Last 10 tokens: [  262 25103 12023   290  8359   257 12625  8073    13 50256]

Sample decoded text:
Today, Tommy was very excited. He was going flying with his mom and dad to a special place. There was a big green flag waving in the wind when they arrived. Tommy was so happy to see it. 

He hopped out of the car and ran up to the flag. He couldn't believe how big it was! He wanted to reach out and touch it. His mom said he could and Tommy smiled.

He waved his little arms, trying to make the flag move. But the wind was too strong. His dad said, "Let me help you, Tommy." He reached out and grabbed the green flag and waved it back and forth. 

Now it was Tommy's turn! He clapped his hands and laughed as he tried to move the flag. He waved it for a long time until he was too tired to do it anymore. The big green flag was so exciti

In [14]:
prepare_data(split="validation", output_file="validation.bin")

validation.bin already exists. Skipping processing.


(memmap([ 7454,  2402,   257, ..., 20567,    13, 50256], dtype=uint16),
 <Encoding 'gpt2'>)

In [15]:
class BatchGenerator:
    def __init__(self, data_file, block_size, batch_size, device):
        self.data_file = data_file
        self.block_size = block_size
        self.batch_size = batch_size
        self.device = device
        self.device_type = 'cuda' if 'cuda' in device.type else 'cpu'

    def get_batch(self, shifted=True):
        # We recreate np.memmap every batch to avoid a memory leak, as per
        # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
        data = np.memmap(self.data_file, dtype=np.uint16, mode='r')

        # Generate random starting indices
        ix = torch.randint(len(data) - self.block_size, (self.batch_size,))

        shift = 1 if shifted else 0
        # Create input and target tensors
        x = torch.stack([torch.from_numpy((data[i:i+self.block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+shift:i+shift+self.block_size]).astype(np.int64)) for i in ix])

        if self.device_type == 'cuda':
            # Pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
            x, y = x.pin_memory().to(self.device, non_blocking=True), y.pin_memory().to(self.device, non_blocking=True)
        else:
            x, y = x.to(self.device), y.to(self.device)

        return x, y

In [17]:
class LossEstimator:
    def __init__(self, model, train_batch_gen, val_batch_gen, eval_iters):
        self.model = model
        self.train_batch_gen = train_batch_gen
        self.val_batch_gen = val_batch_gen
        self.eval_iters = eval_iters

    def estimate_loss(self):
        out = {}
        self.model.eval()
        with torch.inference_mode():
            for split, batch_gen in [('train', self.train_batch_gen), ('val', self.val_batch_gen)]:
                losses = torch.zeros(self.eval_iters)
                for k in tqdm(range(self.eval_iters)):
                    X, Y = batch_gen.get_batch()
                    logits, loss = self.model(X, Y)
                    losses[k] = loss.item()
                out[split] = losses.mean()
        self.model.train()
        return out

    def estimate_loss_pretrainedformat(self):
        out = {}
        self.model.eval()
        with torch.inference_mode():
            for split, batch_gen in [('train', self.train_batch_gen), ('val', self.val_batch_gen)]:
                losses = torch.zeros(self.eval_iters)
                for k in tqdm(range(self.eval_iters)):
                    X, Y = batch_gen.get_batch(shifted=False)
                    ret = self.model(X, labels=Y)
                    losses[k] = ret["loss"].item()
                out[split] = losses.mean()
        self.model.train()
        return out

In [34]:
ts3m.to(device)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 128)
    (wpe): Embedding(2048, 128)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-7): 8 x GPTNeoBlock(
        (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=128, out_features=128, bias=False)
            (v_proj): Linear(in_features=128, out_features=128, bias=False)
            (q_proj): Linear(in_features=128, out_features=128, bias=False)
            (out_proj): Linear(in_features=128, out_features=128, bias=True)
          )
        )
        (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=128, out_features=512, bias=True)
          (c_proj): Linear(in_featu

In [35]:
model

GPTNeo(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 128)
    (wpe): Embedding(2048, 128)
    (h): ModuleList(
      (0-7): 8 x GPTNeoBlock(
        (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (attention): ModuleDict(
            (k_proj): Linear(in_features=128, out_features=128, bias=False)
            (v_proj): Linear(in_features=128, out_features=128, bias=False)
            (q_proj): Linear(in_features=128, out_features=128, bias=False)
            (out_proj): Linear(in_features=128, out_features=128, bias=True)
          )
        )
        (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=128, out_features=512, bias=True)
          (c_proj): Linear(in_features=512, out_features=128, bias=True)
          (act): NewGELUActivation()
        )
      )
    )
    (ln_f): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_

In [20]:
block_size = 256
batch_size = 64
eval_iters = 200

train_batch_gen = BatchGenerator('train.bin', block_size, batch_size, device)
val_batch_gen = BatchGenerator('validation.bin', block_size, batch_size, device)

loss_estimator = LossEstimator(ts3m, train_batch_gen, val_batch_gen, eval_iters)

losses = loss_estimator.estimate_loss_pretrainedformat()

print(f"Estimated train loss: {losses['train']:.4f}")
print(f"Estimated validation loss: {losses['val']:.4f}")

100%|██████████| 200/200 [00:11<00:00, 16.87it/s]

Estimated train loss: 1.9309
Estimated validation loss: 1.9196


In [39]:
block_size = 256
batch_size = 64
eval_iters = 200

model.to(device)

train_batch_gen = BatchGenerator('train.bin', block_size, batch_size, device)
val_batch_gen = BatchGenerator('validation.bin', block_size, batch_size, device)

loss_estimator = LossEstimator(model, train_batch_gen, val_batch_gen, eval_iters)

losses = loss_estimator.estimate_loss()

print(f"Estimated train loss: {losses['train']:.4f}")
print(f"Estimated validation loss: {losses['val']:.4f}")

100%|██████████| 200/200 [00:10<00:00, 18.47it/s]

Estimated train loss: 1.9320
Estimated validation loss: 1.9215


In [40]:
batch = BatchGenerator("train.bin", 256, 64, torch.device("cuda")).get_batch()

In [41]:
ts3m_out = ts3m(batch[0])

In [42]:
my_model_out = model(batch[0])

In [48]:
ts3m_out["logits"][0,-1,:10]

tensor([ 0.3550,  2.4461, -1.0668, -2.3878, -1.3719, -3.2366,  0.6278, -3.7955,
        -2.9903, -5.8510], device='cuda:0', grad_fn=<SliceBackward0>)

In [47]:
my_model_out[0][0,-1,:10]

tensor([ 0.3550,  2.4461, -1.0668, -2.3878, -1.3719, -3.2366,  0.6278, -3.7955,
        -2.9903, -5.8510], device='cuda:0', grad_fn=<SliceBackward0>)

In [56]:
type(ts3m), type(model)

(transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoForCausalLM,
 __main__.GPTNeo)